# Baseline Brute-Force Similarity Search (v2)
## DSA Project: Near-Duplicate and Similarity Search with LSH

This notebook is the **exact baseline** that the MinHash + LSH module will be measured against. It follows the project workflow:

$$\text{Documents} \rightarrow \text{Preprocessing} \rightarrow \text{Shingling} \rightarrow \text{Exact Jaccard} \rightarrow \text{Brute-Force Scan (all pairs / one query)}$$

### What changed from v1, and why

| v1 problem | What we saw | Fix in v2 |
|---|---|---|
| Shingle size `k=9` on stopword-stripped text | Resemblance and containment of **0.00%** even between papers on the same topic (the LSH query returned 0 shared shingles). Nine-word chunks almost never repeat across different papers. | Two **named configurations**: `NEAR_DUP_CFG` (k=3, as in the project document) and `SIMILARITY_CFG` (k=1 content words + Broder's common-shingle filter). |
| Plain k=1 overlap (older run) | ViT-BERT and GAT-Transformer scored as "similar" as same-domain pairs, because all ML papers share generic vocabulary. | Common-shingle filter (`max_df`), rare-shingle filter (`min_df`), references section removed, numbers dropped. |
| Default metric was containment | LSH/MinHash approximates **Jaccard**, not containment. | Jaccard is the default and the ground truth. Containment is kept as an optional extra. |
| Threshold 0.20 for everything | The project document defines near-duplicates at **0.80**. | Separate thresholds: `NEAR_DUP_THRESHOLD = 0.80`, `SIMILARITY_THRESHOLD` (tunable). |
| 17 PDFs only | 136 pairs in about 20 ms, so no speed difference can ever show. | Scaling experiment on passage-level documents, plus the JSON/TXT/dict loader agreed with the extraction team. |
| Query and corpus preprocessed by hand | Easy to mismatch `k` or stopword settings. | One `ShingleConfig` object is stored inside the index and used for both corpus and queries. |
| Unused code, loose claims | Character shingles and Jaccard distance were never used; "exactly 100% accuracy". | Removed. Brute force is the *exact baseline under the chosen representation and threshold*. |

### Near-duplicate vs similarity search (the two tasks in the project statement)
- **Near-duplicate** asks: *is this almost the same text?* Use 3-word shingles and a high threshold (0.80).
- **Similarity search** asks: *which documents are about the same thing?* Whole documents written in different words share few long phrases, so we use single content words (k=1) and filter out generic vocabulary.

Both are still **shingle sets compared with Jaccard**, so MinHash and LSH apply to both.

In [1]:
import os, re, json, time, math, random
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Set, Tuple, Any, Optional

# ---- Dataset location (same auto-detection as v1) ----
if os.path.exists("Research_Papers_Dataset") and len(os.listdir("Research_Papers_Dataset")) > 0:
    DATASET_DIR = "Research_Papers_Dataset"
elif os.path.exists("Sample Dataset") and len(os.listdir("Sample Dataset")) > 0:
    DATASET_DIR = "Sample Dataset"
else:
    DATASET_DIR = "."

MAX_PAGES_PER_DOC = 30      # research papers are typically 8-20 pages
RANDOM_SEED = 42


@dataclass(frozen=True)
class ShingleConfig:
    """Everything that defines how a text becomes a shingle set.
    The SAME config must be used for the corpus, every query, and the LSH module."""
    k: int = 3                       # words per shingle
    remove_stopwords: bool = False
    min_token_len: int = 2           # keeps tokens like 'ai', 'ml'; drops single letters
    drop_numbers: bool = True        # drops pure numbers (page numbers, reference indices)
    strip_references: bool = False   # cut the bibliography (author names are noise for topic)
    max_df: float = 1.0              # drop shingles present in > max_df of documents (Broder's filter). 1.0 = keep all
    min_df: int = 1                  # drop shingles present in < min_df documents. 1 = keep all


# Task 1: "Is this almost the same text?"  (project document: k = 3..5, threshold 0.80)
NEAR_DUP_CFG = ShingleConfig(k=3, remove_stopwords=False, strip_references=False, max_df=1.0, min_df=1)
NEAR_DUP_THRESHOLD = 0.80

# Task 2: "Which documents are about the same thing?"
SIMILARITY_CFG = ShingleConfig(k=1, remove_stopwords=True, strip_references=True, max_df=0.6, min_df=2)
SIMILARITY_THRESHOLD = 0.25   # tunable: the distribution cell below prints a data-driven suggestion

print("Environment configured.")
print(f"Dataset directory     : '{DATASET_DIR}'")
print(f"Near-duplicate config : {NEAR_DUP_CFG}")
print(f"Similarity config     : {SIMILARITY_CFG}")

Environment configured.
Dataset directory     : 'Research_Papers_Dataset'
Near-duplicate config : ShingleConfig(k=3, remove_stopwords=False, min_token_len=2, drop_numbers=True, strip_references=False, max_df=1.0, min_df=1)
Similarity config     : ShingleConfig(k=1, remove_stopwords=True, min_token_len=2, drop_numbers=True, strip_references=True, max_df=0.6, min_df=2)


### Step 1: Loading the corpus (agreed handoff formats)
The extraction teammates can hand over text in several forms. `load_corpus` accepts all of them and always returns `{doc_id: text}`:
a Python **dict**, a **JSON file** (`[{"doc_id": ..., "text": ...}]`), or a **folder** of `.pdf` and/or `.txt` files.

In [2]:
def extract_text_from_pdf(pdf_path: str, max_pages: Optional[int] = None) -> str:
    """Page-by-page text extraction with pypdf (imported lazily so JSON/TXT users do not need it)."""
    from pypdf import PdfReader
    try:
        reader = PdfReader(pdf_path)
        n_pages = len(reader.pages)
        limit = min(n_pages, max_pages) if max_pages is not None else n_pages
        pages = []
        for i in range(limit):
            t = reader.pages[i].extract_text()
            if t:
                pages.append(t)
        return "\n".join(pages).strip()      # newline kept: it lets us repair hyphenated line breaks
    except Exception as exc:
        print(f"[ERROR] Could not extract {pdf_path}: {exc}")
        return ""


def load_from_json(path) -> Dict[str, str]:
    with open(path, "r", encoding="utf-8") as f:
        records = json.load(f)
    if isinstance(records, dict):
        return {str(k): str(v) for k, v in records.items()}
    return {str(r["doc_id"]): str(r["text"]) for r in records}


def load_from_directory(directory, max_pages_per_doc: Optional[int] = None) -> Dict[str, str]:
    directory = Path(directory)
    corpus = {}
    for p in sorted(directory.iterdir()):
        suffix = p.suffix.lower()
        if suffix == ".pdf":
            text = extract_text_from_pdf(str(p), max_pages=max_pages_per_doc)
        elif suffix == ".txt":
            text = p.read_text(encoding="utf-8", errors="ignore")
        else:
            continue
        if text.strip():
            corpus[p.name] = text
        else:
            print(f"[WARNING] Skipping '{p.name}': no extractable text (scanned/image-only?).")
    return corpus


def load_corpus(source, max_pages_per_doc: Optional[int] = None) -> Dict[str, str]:
    """Dispatch on whatever the extraction team hands over: dict | .json file | folder of .pdf/.txt."""
    if isinstance(source, dict):
        return {str(k): str(v) for k, v in source.items()}
    path = Path(source)
    if path.is_dir():
        return load_from_directory(path, max_pages_per_doc)
    if path.suffix.lower() == ".json":
        return load_from_json(path)
    raise ValueError(f"Unsupported corpus source: {source!r}")


t0 = time.perf_counter()
corpus = load_corpus(DATASET_DIR, max_pages_per_doc=MAX_PAGES_PER_DOC)
print(f"Loaded {len(corpus)} documents from '{DATASET_DIR}' in {time.perf_counter() - t0:.2f} s\n")

print("=" * 95)
print(f"{'Document':<62} | {'Characters':<12} | {'Words':<8}")
print("=" * 95)
for doc_id, text in corpus.items():
    name = (doc_id[:59] + '..') if len(doc_id) > 61 else doc_id
    print(f"{name:<62} | {len(text):<12} | {len(text.split()):<8}")
print("=" * 95)

Loaded 17 documents from 'Research_Papers_Dataset' in 18.87 s

Document                                                       | Characters   | Words   
CV_2014_Simonyan_VGG_Deep_Convolutional_Networks.pdf           | 54984        | 9008    
CV_2015_He_ResNet_Deep_Residual_Learning.pdf                   | 59422        | 9803    
CV_2020_Dosovitskiy_Vision_Transformer_ViT.pdf                 | 67280        | 10574   
GRAPH_2016_Kipf_Semi_Supervised_GCN.pdf                        | 43282        | 6862    
GRAPH_2017_Hamilton_Inductive_Representation_GraphSAGE.pdf     | 69717        | 10849   
GRAPH_2017_Velickovic_Graph_Attention_Networks_GAT.pdf         | 43462        | 6457    
LSH_2014_Wang_Hashing_for_Similarity_Search_Survey.pdf         | 147117       | 25699   
LSH_2018_Echihabi_Benchmark_High_Dim_NN.pdf                    | 78103        | 14348   
LSH_2019_Aumuller_ANN_Benchmarks.pdf                           | 65724        | 10201   
LSH_2020_Sundaram_LSH_for_Genomics.pdf         

### Step 2: Preprocessing (one function, driven by the config)
1. Fix typographic ligatures (`ﬁ` to `fi`) and hyphenated line breaks (`simi-` / `larity` to `similarity`).
2. Lowercase and keep only letters and digits.
3. Optional: **cut the references section** (a bibliography is full of author names and years that say nothing about topic).
4. Drop pure numbers, very short tokens, and (optionally) stopwords.

In [3]:
STOPWORDS = {
    'the', 'a', 'an', 'and', 'or', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by',
    'from', 'up', 'about', 'into', 'over', 'after', 'is', 'are', 'was', 'were', 'be',
    'been', 'being', 'have', 'has', 'had', 'do', 'does', 'did', 'this', 'that', 'these',
    'those', 'it', 'its', 'as', 'we', 'you', 'they', 'he', 'she', 'which', 'who', 'what',
    'where', 'when', 'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more', 'most',
    'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than',
    'too', 'very', 'can', 'will', 'just', 'should', 'now', 'our', 'their', 'then', 'there',
    'et', 'al', 'fig', 'figure', 'table', 'section', 'also', 'using', 'used', 'use', 'may',
    'however', 'thus', 'since', 'while', 'where', 'whose', 'per', 'via', 'et'
}

_LIGATURES = {'\ufb00': 'ff', '\ufb01': 'fi', '\ufb02': 'fl', '\ufb03': 'ffi', '\ufb04': 'ffl',
              '\u2019': "'", '\u201c': '"', '\u201d': '"', '\u2014': ' ', '\u2013': ' ', '\u00ad': ''}


def clean_text(text: str, strip_references: bool = False) -> str:
    for old, new in _LIGATURES.items():
        text = text.replace(old, new)
    text = text.lower()
    text = re.sub(r"-\s*\n\s*", "", text)                 # simi-\nlarity -> similarity
    if strip_references:
        hits = list(re.finditer(r"\breferences\b|\bbibliography\b", text))
        if hits and hits[-1].start() > 0.5 * len(text):   # only cut if it is in the back half
            text = text[:hits[-1].start()]
    return text


def preprocess_text(text: str, cfg: ShingleConfig) -> List[str]:
    """text -> list of tokens, according to cfg."""
    text = clean_text(text, cfg.strip_references)
    tokens = re.findall(r"[a-z0-9]+", text)
    out = []
    for w in tokens:
        if len(w) < cfg.min_token_len:
            continue
        if cfg.drop_numbers and w.isdigit():
            continue
        if cfg.remove_stopwords and w in STOPWORDS:
            continue
        out.append(w)
    return out


def build_shingles(tokens: List[str], k: int) -> Set[str]:
    """k-word shingles: ['a','b','c','d'], k=2 -> {'a b', 'b c', 'c d'}"""
    if not tokens:
        return set()
    if len(tokens) < k:
        return {" ".join(tokens)}
    return {" ".join(tokens[i:i + k]) for i in range(len(tokens) - k + 1)}


demo = "Machine Learning is VERY useful! Machine learning is useful."
print("tokens   :", preprocess_text(demo, NEAR_DUP_CFG))
print("shingles :", sorted(build_shingles(preprocess_text(demo, NEAR_DUP_CFG), 3)))

tokens   : ['machine', 'learning', 'is', 'very', 'useful', 'machine', 'learning', 'is', 'useful']
shingles : ['is very useful', 'learning is useful', 'learning is very', 'machine learning is', 'useful machine learning', 'very useful machine']


### Step 3: Exact similarity measures (pure set operations)
- **Jaccard resemblance**: $J(A,B)=\dfrac{|A\cap B|}{|A\cup B|}$. Symmetric, 0 to 1. **This is what MinHash estimates.**
- **Containment**: $c(A,B)=\dfrac{|A\cap B|}{|A|}$. Asymmetric. Useful when a short excerpt is compared with a long document (Jaccard is small there because the union is dominated by the long document).
- **Max containment**: $\dfrac{|A\cap B|}{\min(|A|,|B|)}$. Detects "one document is inside the other".

In [4]:
def jaccard_resemblance(a: Set[str], b: Set[str]) -> float:
    """|A n B| / |A u B|.  Two empty sets give 0.0 (nothing to compare)."""
    if not a or not b:
        return 0.0
    inter = len(a & b)
    return inter / (len(a) + len(b) - inter)


def containment(a: Set[str], b: Set[str]) -> float:
    """|A n B| / |A|: how much of A appears in B."""
    return len(a & b) / len(a) if a else 0.0


def max_containment(a: Set[str], b: Set[str]) -> float:
    """|A n B| / min(|A|,|B|)."""
    if not a or not b:
        return 0.0
    return len(a & b) / min(len(a), len(b))


# ---- unit tests ----
A, B, C = {"a", "b", "c", "d"}, {"b", "c", "d", "e"}, {"x", "y"}
assert math.isclose(jaccard_resemblance(A, B), 3 / 5)          # the worked example from the project document
assert jaccard_resemblance(A, A) == 1.0 and jaccard_resemblance(A, C) == 0.0
assert jaccard_resemblance(A, B) == jaccard_resemblance(B, A)
assert math.isclose(containment({"a", "b"}, A), 1.0) and math.isclose(containment(A, {"a", "b"}), 0.5)
assert max_containment({"a", "b"}, A) == 1.0
print("All similarity-measure tests passed.")

All similarity-measure tests passed.


### Step 4: The brute-force engine
`BruteForceIndex` stores the shingle sets and the config. It has **no indexing, no pruning, no shortcuts**: every query scans every document, and `all_pairs` performs exactly $N(N-1)/2$ comparisons.
It deliberately exposes the same methods the LSH index will expose (`query`, `most_similar`, `near_duplicates_of`, `all_pairs`, `stats`), so the frontend can swap one for the other.

The **common-shingle filter** (Broder et al., *Syntactic Clustering of the Web*) removes shingles that occur in more than `max_df` of the documents. In research papers these are generic words such as "model" or "training": they create false resemblance between unrelated papers. `min_df=2` drops words that occur in only one document, because they can never match anything and only inflate the union.

In [5]:
class BruteForceIndex:
    """Exact, exhaustive similarity search over shingle sets. This is the ground truth for LSH."""

    def __init__(self, corpus: Dict[str, str], cfg: ShingleConfig):
        self.cfg = cfg
        self.doc_ids: List[str] = list(corpus.keys())
        self.n_docs = len(self.doc_ids)

        raw = {d: build_shingles(preprocess_text(t, cfg), cfg.k) for d, t in corpus.items()}
        self.df: Counter = Counter()
        for s in raw.values():
            self.df.update(s)
        self.shingles: Dict[str, Set[str]] = {d: {sh for sh in s if self._allowed(sh)} for d, s in raw.items()}

        self.last_comparisons = 0
        self.last_elapsed = 0.0

    # ---- common / rare shingle filter (applied identically to corpus and queries) ----
    def _allowed(self, shingle: str) -> bool:
        d = self.df.get(shingle, 0)
        if self.cfg.max_df < 1.0 and d > self.cfg.max_df * self.n_docs:
            return False
        if self.cfg.min_df > 1 and d < self.cfg.min_df:
            return False
        return True

    def shingle_text(self, text: str) -> Set[str]:
        sh = build_shingles(preprocess_text(text, self.cfg), self.cfg.k)
        return {s for s in sh if self._allowed(s)}

    # ---- one query against the whole corpus ----
    def query(self, query_text: str, measure: str = "jaccard", top_n: Optional[int] = 5,
              exclude_id: Optional[str] = None) -> Dict[str, Any]:
        q = self.shingle_text(query_text)
        t0 = time.perf_counter()
        rows = []
        for doc_id in self.doc_ids:
            if doc_id == exclude_id:
                continue
            d = self.shingles[doc_id]
            jac, cont = jaccard_resemblance(q, d), containment(q, d)
            rows.append({"id": doc_id, "score": jac if measure == "jaccard" else cont,
                         "jaccard": jac, "containment": cont,
                         "shared": len(q & d), "doc_shingles": len(d)})
        # rank by the chosen measure; break ties with Jaccard (fixes the 100%-containment ties seen in v1)
        rows.sort(key=lambda r: (-r["score"], -r["jaccard"], r["id"]))
        self.last_elapsed = time.perf_counter() - t0
        self.last_comparisons = len(rows)
        return {"results": rows[:top_n] if top_n else rows, "comparisons": len(rows),
                "elapsed_seconds": self.last_elapsed, "query_shingles": len(q), "measure": measure}

    def most_similar(self, query_text: str, measure: str = "jaccard"):
        res = self.query(query_text, measure=measure, top_n=1)["results"]
        return res[0] if res else None

    def near_duplicates_of(self, query_text: str, threshold: float = NEAR_DUP_THRESHOLD, measure: str = "jaccard"):
        res = self.query(query_text, measure=measure, top_n=None)["results"]
        return [r for r in res if r["score"] >= threshold]

    # ---- all pairs inside the corpus (the ground truth for LSH) ----
    def all_pairs(self, threshold: float, keep_scores: bool = True) -> Dict[str, Any]:
        ids, S = self.doc_ids, self.shingles
        n = len(ids)
        pairs, scores, comparisons = [], [], 0
        t0 = time.perf_counter()
        for i in range(n):
            a = S[ids[i]]
            for j in range(i + 1, n):
                s = jaccard_resemblance(a, S[ids[j]])
                comparisons += 1
                if keep_scores:
                    scores.append(s)
                if s >= threshold:
                    pairs.append({"doc_a": ids[i], "doc_b": ids[j], "jaccard": round(s, 4),
                                  "shared": len(a & S[ids[j]])})
        self.last_elapsed = time.perf_counter() - t0
        self.last_comparisons = comparisons
        pairs.sort(key=lambda p: -p["jaccard"])
        return {"similar_pairs": pairs, "total_comparisons": comparisons,
                "elapsed_seconds": self.last_elapsed, "threshold": threshold, "all_scores": scores}

    def theoretical_comparisons(self) -> int:
        return self.n_docs * (self.n_docs - 1) // 2

    def stats(self) -> Dict[str, Any]:
        sizes = [len(s) for s in self.shingles.values()] or [0]
        return {"num_documents": self.n_docs, "avg_shingles_per_doc": round(sum(sizes) / len(sizes), 1),
                "comparisons_performed": self.last_comparisons,
                "elapsed_seconds": round(self.last_elapsed, 6),
                "comparisons_per_second": round(self.last_comparisons / self.last_elapsed, 1) if self.last_elapsed > 0 else None}


def print_results(title: str, out: Dict[str, Any]):
    print("\n" + "=" * 104)
    print(f"QUERY: {title}")
    print(f"Compared with {out['comparisons']} documents | query shingles: {out['query_shingles']} | "
          f"scan time: {out['elapsed_seconds'] * 1000:.2f} ms | ranked by: {out['measure'].upper()}")
    print("=" * 104)
    print(f"{'Rank':<5} | {'Jaccard':<9} | {'Containment':<12} | {'Shared':<7} | Document")
    print("-" * 104)
    for i, r in enumerate(out["results"], 1):
        name = (r['id'][:60] + '..') if len(r['id']) > 62 else r['id']
        print(f"{i:<5} | {r['jaccard'] * 100:>7.2f}% | {r['containment'] * 100:>10.2f}% | {r['shared']:<7} | {name}")
    print("=" * 104)
    if out["query_shingles"] == 0:
        print("NOTE: no query shingle survived the filters (generic words are dropped by design), so every score is 0.")


sim_index = BruteForceIndex(corpus, SIMILARITY_CFG)    # task 2: topical similarity
dup_index = BruteForceIndex(corpus, NEAR_DUP_CFG)      # task 1: near-duplicates
print("Similarity index :", sim_index.stats())
print("Near-dup index   :", dup_index.stats())

Similarity index : {'num_documents': 17, 'avg_shingles_per_doc': 756.0, 'comparisons_performed': 0, 'elapsed_seconds': 0.0, 'comparisons_per_second': None}
Near-dup index   : {'num_documents': 17, 'avg_shingles_per_doc': 7944.9, 'comparisons_performed': 0, 'elapsed_seconds': 0.0, 'comparisons_per_second': None}


### Step 5: Diagnostic. Which settings give *meaningful* similarity?
If the papers carry a domain prefix in their file name (`CV_`, `GRAPH_`, `LSH_`, `NLP_`, `SEC_`), we have free ground truth for topical similarity. A good configuration should give

- **same-domain** pairs a clearly higher Jaccard than **cross-domain** pairs (look at the *ratio*), and
- a high **top-1 accuracy** (the closest document is from the same domain).

The table also shows why v1 failed: the `k=9` row has near-zero similarity for *everything*, so nothing can be separated.

In [6]:
def domain_of(doc_id: str) -> Optional[str]:
    """'CV_2015_He_ResNet.pdf' -> 'CV'. Returns None if the name has no prefix."""
    return doc_id.split("_")[0] if "_" in doc_id else None


def separation_report(index: BruteForceIndex) -> Optional[Dict[str, float]]:
    ids, S = index.doc_ids, index.shingles
    labels = [domain_of(d) for d in ids]
    if None in labels or len(set(labels)) < 2:
        return None
    n = len(ids)
    score = {}
    for i in range(n):
        for j in range(i + 1, n):
            score[(i, j)] = score[(j, i)] = jaccard_resemblance(S[ids[i]], S[ids[j]])
    same = [score[(i, j)] for i in range(n) for j in range(i + 1, n) if labels[i] == labels[j]]
    cross = [score[(i, j)] for i in range(n) for j in range(i + 1, n) if labels[i] != labels[j]]
    hits = tot = 0
    for i in range(n):
        if not any(labels[j] == labels[i] for j in range(n) if j != i):
            continue
        best = max((j for j in range(n) if j != i), key=lambda j: score[(i, j)])
        tot += 1
        hits += labels[best] == labels[i]
    mean = lambda x: sum(x) / len(x) if x else 0.0
    return {"same": mean(same), "cross": mean(cross), "ratio": mean(same) / mean(cross) if mean(cross) > 0 else float('inf'),
            "top1": hits / tot if tot else 0.0}


SWEEP = [
    ("k=9  stopwords, no filter  (v1 setting)", ShingleConfig(k=9, remove_stopwords=True, strip_references=False)),
    ("k=3  no stopwords          (near-dup)",   NEAR_DUP_CFG),
    ("k=2  stopwords, df filter",               ShingleConfig(k=2, remove_stopwords=True, strip_references=True, max_df=0.6, min_df=2)),
    ("k=1  stopwords, no filter  (plain)",      ShingleConfig(k=1, remove_stopwords=True, strip_references=False)),
    ("k=1  + refs removed",                     ShingleConfig(k=1, remove_stopwords=True, strip_references=True)),
    ("k=1  + refs + max_df=0.8, min_df=2",      ShingleConfig(k=1, remove_stopwords=True, strip_references=True, max_df=0.8, min_df=2)),
    ("k=1  + refs + max_df=0.6, min_df=2  <-- SIMILARITY_CFG", SIMILARITY_CFG),
    ("k=1  + refs + max_df=0.5, min_df=2",      ShingleConfig(k=1, remove_stopwords=True, strip_references=True, max_df=0.5, min_df=2)),
]

print(f"{'Configuration':<58} | {'avg sh/doc':>10} | {'same-dom J':>10} | {'cross-dom J':>11} | {'ratio':>6} | {'top-1':>6}")
print("-" * 118)
for name, cfg in SWEEP:
    idx = BruteForceIndex(corpus, cfg)
    rep = separation_report(idx)
    avg = idx.stats()["avg_shingles_per_doc"]
    if rep is None:
        print(f"{name:<58} | {avg:>10} | (no domain prefixes in file names: cannot score)")
    else:
        print(f"{name:<58} | {avg:>10} | {rep['same']:>10.4f} | {rep['cross']:>11.4f} | {rep['ratio']:>6.1f} | {rep['top1']*100:>5.0f}%")

Configuration                                              | avg sh/doc | same-dom J | cross-dom J |  ratio |  top-1
----------------------------------------------------------------------------------------------------------------------
k=9  stopwords, no filter  (v1 setting)                    |     5887.4 |     0.0016 |      0.0002 |   10.2 |    59%
k=3  no stopwords          (near-dup)                      |     7944.9 |     0.0129 |      0.0065 |    2.0 |    71%
k=2  stopwords, df filter                                  |      399.6 |     0.1013 |      0.0411 |    2.5 |    76%
k=1  stopwords, no filter  (plain)                         |     1700.2 |     0.1723 |      0.1527 |    1.1 |    71%
k=1  + refs removed                                        |     1323.4 |     0.1684 |      0.1537 |    1.1 |    59%
k=1  + refs + max_df=0.8, min_df=2                         |      886.2 |     0.2039 |      0.1755 |    1.2 |    65%
k=1  + refs + max_df=0.6, min_df=2  <-- SIMILARITY_CFG     |  

### Step 6: Experiment 1. Topical similarity (leave-one-out)
Each paper is used as a query against all the *other* papers. Related papers should now rank first with a clearly non-zero Jaccard.

In [7]:
rows_summary, hits, evaluable = [], 0, 0
for doc_id in corpus:
    out = sim_index.query(corpus[doc_id], measure="jaccard", top_n=3, exclude_id=doc_id)
    top = out["results"][0]
    same_domain = (domain_of(top["id"]) == domain_of(doc_id)) if domain_of(doc_id) else None
    rows_summary.append((doc_id, top, same_domain))
    if same_domain is not None and any(domain_of(o) == domain_of(doc_id) for o in corpus if o != doc_id):
        evaluable += 1
        hits += bool(same_domain)

print(f"{'Query document':<52} | {'Closest document':<52} | {'Jaccard':>8} | Same domain?")
print("-" * 132)
for doc_id, top, same in rows_summary:
    q = (doc_id[:49] + '..') if len(doc_id) > 51 else doc_id
    t = (top['id'][:49] + '..') if len(top['id']) > 51 else top['id']
    flag = "-" if same is None else ("yes" if same else "NO")
    print(f"{q:<52} | {t:<52} | {top['jaccard']*100:>7.2f}% | {flag}")
if evaluable:
    print(f"\nTop-1 same-domain accuracy: {hits}/{evaluable} = {100 * hits / evaluable:.0f}%")

# Three detailed examples
for doc_id in list(corpus)[:: max(1, len(corpus) // 3)][:3]:
    print_results(f"{doc_id} (leave-one-out, Jaccard)", sim_index.query(corpus[doc_id], top_n=3, exclude_id=doc_id))

Query document                                       | Closest document                                     |  Jaccard | Same domain?
------------------------------------------------------------------------------------------------------------------------------------
CV_2014_Simonyan_VGG_Deep_Convolutional_Networks...  | CV_2015_He_ResNet_Deep_Residual_Learning.pdf         |   19.71% | yes
CV_2015_He_ResNet_Deep_Residual_Learning.pdf         | CV_2014_Simonyan_VGG_Deep_Convolutional_Networks...  |   19.71% | yes
CV_2020_Dosovitskiy_Vision_Transformer_ViT.pdf       | NLP_2019_Raffel_T5_Transfer_Learning.pdf             |   25.80% | NO
GRAPH_2016_Kipf_Semi_Supervised_GCN.pdf              | GRAPH_2017_Velickovic_Graph_Attention_Networks_GA..  |   23.56% | yes
GRAPH_2017_Hamilton_Inductive_Representation_Grap..  | GRAPH_2017_Velickovic_Graph_Attention_Networks_GA..  |   22.50% | yes
GRAPH_2017_Velickovic_Graph_Attention_Networks_GA..  | GRAPH_2016_Kipf_Semi_Supervised_GCN.pdf              |

### Step 7: Experiment 2. Near-duplicate detection (k=3, threshold 0.80)
We take a whole paper and damage it with random deletions, substitutions and insertions at different **edit rates**. Each edited word breaks about 3 of the 3-word shingles, so Jaccard falls smoothly as the edit rate grows. This shows *where* the 0.80 threshold sits.

Then we compare a **short excerpt** with the full paper, to show why Jaccard is the wrong measure for excerpts (containment is the right one).

In [8]:
def make_near_duplicate(text: str, edit_rate: float = 0.05, seed: int = 0) -> str:
    """Randomly delete / substitute / insert words (each with probability edit_rate/3)."""
    rng = random.Random(seed)
    words = text.split()
    vocab = sorted(set(words))      # sorted: reproducible with the seed
    out = []
    for w in words:
        r = rng.random()
        if r < edit_rate / 3:
            continue                                   # deletion
        elif r < 2 * edit_rate / 3:
            out.append(rng.choice(vocab))              # substitution
        elif r < edit_rate:
            out.extend([w, rng.choice(vocab)])         # insertion
        else:
            out.append(w)
    return " ".join(out)


target_id = list(corpus)[0]
print(f"Source document: {target_id}\n")
print(f"{'Edit rate':<10} | {'Jaccard (k=3)':<14} | {'Retrieved rank-1 is the source?':<32} | Near-duplicate at {NEAR_DUP_THRESHOLD}?")
print("-" * 95)
for rate in [0.01, 0.03, 0.05, 0.10, 0.20]:
    perturbed = make_near_duplicate(corpus[target_id], rate, seed=RANDOM_SEED)
    top = dup_index.query(perturbed, measure="jaccard", top_n=2)["results"][0]
    ok = top["id"] == target_id
    print(f"{rate*100:>7.0f}%   | {top['jaccard']*100:>12.2f}% | {str(ok):<32} | {'YES' if top['jaccard'] >= NEAR_DUP_THRESHOLD else 'no'}")
    assert ok, "the damaged copy must still retrieve its own source at rank 1"

# Excerpt vs full document
_w = corpus[target_id].split()
_start = max(0, len(_w) // 2 - 200)
excerpt = " ".join(_w[_start:_start + 400])                  # a ~400-word passage from the middle
out = dup_index.query(excerpt, measure="containment", top_n=3)
print_results("400-word excerpt of the source paper (ranked by CONTAINMENT)", out)
top = out["results"][0]
print(f"\nExcerpt vs source: containment = {top['containment']*100:.1f}%  but Jaccard = {top['jaccard']*100:.1f}%  "
      "(the union is dominated by the long paper), so excerpts need containment, near-duplicate pairs need Jaccard.")

Source document: CV_2014_Simonyan_VGG_Deep_Convolutional_Networks.pdf

Edit rate  | Jaccard (k=3)  | Retrieved rank-1 is the source?  | Near-duplicate at 0.8?
-----------------------------------------------------------------------------------------------
      1%   |        90.67% | True                             | YES
      3%   |        82.60% | True                             | YES
      5%   |        75.29% | True                             | no
     10%   |        59.69% | True                             | no
     20%   |        36.89% | True                             | no

QUERY: 400-word excerpt of the source paper (ranked by CONTAINMENT)
Compared with 17 documents | query shingles: 312 | scan time: 0.60 ms | ranked by: CONTAINMENT
Rank  | Jaccard   | Containment  | Shared  | Document
--------------------------------------------------------------------------------------------------------
1     |    4.51% |      96.79% | 302     | CV_2014_Simonyan_VGG_Deep_Convolutional_Ne

### Step 8: Experiment 3. Free-form keyword queries
A short keyword query is tiny compared with a paper, so we rank by **containment** (how much of the query appears in the paper), with Jaccard as tie-breaker. In v1 many papers tied at 100% containment; the tie-breaker now separates them. Generic words that appear in most papers (for example "neural" or "network") are removed by the common-shingle filter, so the query is judged on its *distinctive* words; `query shingles` shows how many survived.

In [9]:
free_queries = [
    "locality sensitive hashing approximate nearest neighbor search high dimensional minhash collision probability",
    "transformer self attention mechanism multi head attention language model pre training",
    "graph convolutional networks node classification neighborhood aggregation spectral convolution",
    "deep neural network software vulnerability detection source code graph identification",
]
for q in free_queries:
    print_results(f"'{q[:70]}...'", sim_index.query(q, measure="containment", top_n=3))


QUERY: 'locality sensitive hashing approximate nearest neighbor search high di...'
Compared with 17 documents | query shingles: 10 | scan time: 0.10 ms | ranked by: CONTAINMENT
Rank  | Jaccard   | Containment  | Shared  | Document
--------------------------------------------------------------------------------------------------------
1     |    0.94% |     100.00% | 10      | LSH_2021_Zheng_LSH_Survey_and_Applications.pdf
2     |    0.80% |     100.00% | 10      | LSH_2014_Wang_Hashing_for_Similarity_Search_Survey.pdf
3     |    0.89% |      80.00% | 8       | LSH_2019_Aumuller_ANN_Benchmarks.pdf

QUERY: 'transformer self attention mechanism multi head attention language mod...'
Compared with 17 documents | query shingles: 6 | scan time: 0.09 ms | ranked by: CONTAINMENT
Rank  | Jaccard   | Containment  | Shared  | Document
--------------------------------------------------------------------------------------------------------
1     |    0.96% |     100.00% | 6       | NLP_2017_Vaswani

### Step 9: Experiment 4. All-pairs ground truth (exact, $N(N-1)/2$ comparisons)
Two ground-truth sets are produced for the LSH evaluation:

1. **Similarity ground truth**: all pairs of papers with Jaccard at least `SIMILARITY_THRESHOLD` (under `SIMILARITY_CFG`).
2. **Near-duplicate ground truth**: distinct papers are *not* near-duplicates of each other (0 pairs is the correct answer), so we **plant** damaged copies of a few papers into an evaluation corpus. The planted pairs are the known answers, and LSH must find them.

Both are saved to disk so the LSH notebook can load exactly the same data.

In [10]:
# ---------- 1) similarity ground truth ----------
res_sim = sim_index.all_pairs(threshold=SIMILARITY_THRESHOLD)
scores = sorted(res_sim["all_scores"])
pct = lambda p: scores[min(len(scores) - 1, int(p / 100 * len(scores)))] if scores else 0.0
print("=" * 100)
print(f"SIMILARITY ground truth | {res_sim['total_comparisons']} pairs compared in {res_sim['elapsed_seconds']*1000:.1f} ms | "
      f"pairs with Jaccard >= {SIMILARITY_THRESHOLD}: {len(res_sim['similar_pairs'])}")
print(f"Pair-score distribution: median={pct(50):.3f}  p75={pct(75):.3f}  p90={pct(90):.3f}  max={scores[-1] if scores else 0:.3f}")
print(f"Suggested threshold (top ~10% of pairs): {pct(90):.3f}   <- set SIMILARITY_THRESHOLD to taste")
print("=" * 100)
for i, p in enumerate(res_sim["similar_pairs"][:10], 1):
    a = (p['doc_a'][:42] + '..') if len(p['doc_a']) > 44 else p['doc_a']
    b = (p['doc_b'][:42] + '..') if len(p['doc_b']) > 44 else p['doc_b']
    print(f"{i:<3} | {p['jaccard']*100:6.2f}% | {a}  <->  {b}")

# ---------- 2) near-duplicate ground truth with planted copies ----------
eval_corpus = dict(corpus)
planted = []
for doc_id, rate in zip(list(corpus)[:4], [0.01, 0.02, 0.03, 0.05]):
    new_id = f"{doc_id}__dup{int(rate * 100)}pct"
    eval_corpus[new_id] = make_near_duplicate(corpus[doc_id], rate, seed=RANDOM_SEED)
    planted.append((doc_id, new_id, rate))

eval_dup_index = BruteForceIndex(eval_corpus, NEAR_DUP_CFG)
res_dup = eval_dup_index.all_pairs(threshold=NEAR_DUP_THRESHOLD)
print("\n" + "=" * 100)
print(f"NEAR-DUPLICATE ground truth | corpus = {len(corpus)} originals + {len(planted)} planted copies = {len(eval_corpus)} documents")
print(f"{res_dup['total_comparisons']} pairs compared in {res_dup['elapsed_seconds']*1000:.1f} ms | pairs with Jaccard >= {NEAR_DUP_THRESHOLD}: {len(res_dup['similar_pairs'])}")
print("=" * 100)
found = {tuple(sorted((p['doc_a'], p['doc_b']))): p['jaccard'] for p in res_dup['similar_pairs']}
for src, dup, rate in planted:
    key = tuple(sorted((src, dup)))
    j = jaccard_resemblance(eval_dup_index.shingles[src], eval_dup_index.shingles[dup])
    print(f"planted {rate*100:>3.0f}% edits | Jaccard = {j*100:5.1f}% | in ground truth: {'YES' if key in found else 'no (below threshold)'}")

# ---------- save for the LSH notebook ----------
with open("eval_corpus.json", "w", encoding="utf-8") as f:
    json.dump([{"doc_id": d, "text": t} for d, t in eval_corpus.items()], f)
for fname, res, cfg in [("ground_truth_similarity.json", res_sim, SIMILARITY_CFG),
                        ("ground_truth_near_duplicate.json", res_dup, NEAR_DUP_CFG)]:
    with open(fname, "w", encoding="utf-8") as f:
        json.dump({"config": cfg.__dict__, "threshold": res["threshold"], "total_comparisons": res["total_comparisons"],
                   "pairs": [[p["doc_a"], p["doc_b"], p["jaccard"]] for p in res["similar_pairs"]]}, f)
print("\nSaved: eval_corpus.json, ground_truth_similarity.json, ground_truth_near_duplicate.json")

SIMILARITY ground truth | 136 pairs compared in 3.9 ms | pairs with Jaccard >= 0.25: 4
Pair-score distribution: median=0.134  p75=0.165  p90=0.215  max=0.324
Suggested threshold (top ~10% of pairs): 0.215   <- set SIMILARITY_THRESHOLD to taste
1   |  32.38% | LSH_2014_Wang_Hashing_for_Similarity_Searc..  <->  LSH_2021_Zheng_LSH_Survey_and_Applications..
2   |  26.02% | NLP_2018_Devlin_BERT_Pretraining.pdf  <->  NLP_2019_Raffel_T5_Transfer_Learning.pdf
3   |  25.80% | CV_2020_Dosovitskiy_Vision_Transformer_ViT..  <->  NLP_2019_Raffel_T5_Transfer_Learning.pdf
4   |  25.59% | NLP_2019_Raffel_T5_Transfer_Learning.pdf  <->  SEC_2020_Chakraborty_Deep_Learning_Softwar..

NEAR-DUPLICATE ground truth | corpus = 17 originals + 4 planted copies = 21 documents
210 pairs compared in 68.7 ms | pairs with Jaccard >= 0.8: 2
planted   1% edits | Jaccard =  90.7% | in ground truth: YES
planted   2% edits | Jaccard =  76.4% | in ground truth: no (below threshold)
planted   3% edits | Jaccard =  81.7% | i

### Step 10: Experiment 5. Why brute force does not scale
With 17 papers the scan is instant, so no speed-up could ever be shown. We therefore split the papers into **120-word passages** (hundreds of short documents) and time the exact all-pairs scan for growing $N$. The number of comparisons grows as $N(N-1)/2$, so doubling $N$ roughly quadruples the time. The projection at the bottom is what LSH has to beat.

In [11]:
PASSAGE_WORDS = 120
passages = {}
for doc_id, text in corpus.items():
    w = text.split()
    for i in range(0, len(w) - PASSAGE_WORDS + 1, PASSAGE_WORDS):
        passages[f"{doc_id}#p{i // PASSAGE_WORDS}"] = " ".join(w[i:i + PASSAGE_WORDS])
print(f"{len(passages)} passages of {PASSAGE_WORDS} words")

rng = random.Random(RANDOM_SEED)
ids_all = list(passages)
rng.shuffle(ids_all)
sizes = sorted({s for s in (100, 200, 400, 800, 1600) if s <= len(ids_all)} | {min(len(ids_all), 1600)})

print(f"\n{'N docs':>8} | {'pairs N(N-1)/2':>15} | {'scan time (s)':>13} | {'microsec / pair':>15} | {'pairs >= threshold':>18}")
print("-" * 82)
scaling_rows = []
for n in sizes:
    sub = {d: passages[d] for d in ids_all[:n]}
    idx = BruteForceIndex(sub, NEAR_DUP_CFG)
    r = idx.all_pairs(threshold=NEAR_DUP_THRESHOLD, keep_scores=False)
    scaling_rows.append((n, r["total_comparisons"], r["elapsed_seconds"]))
    print(f"{n:>8} | {r['total_comparisons']:>15,} | {r['elapsed_seconds']:>13.3f} | {r['elapsed_seconds'] / r['total_comparisons'] * 1e6:>15.2f} | {len(r['similar_pairs']):>18}")

if scaling_rows:
    per_pair = scaling_rows[-1][2] / scaling_rows[-1][1]
    print("\nProjection with the measured cost per pair (short documents, so real texts will be slower):")
    for n in (1_000, 10_000, 100_000, 1_000_000):
        pairs = n * (n - 1) // 2
        secs = pairs * per_pair
        print(f"  N = {n:>9,}: {pairs:>18,} pairs -> about {secs:>14,.0f} s  ({secs / 3600:,.1f} hours)")

1537 passages of 120 words

  N docs |  pairs N(N-1)/2 | scan time (s) | microsec / pair | pairs >= threshold
----------------------------------------------------------------------------------
     100 |           4,950 |         0.012 |            2.43 |                  0
     200 |          19,900 |         0.051 |            2.56 |                  0
     400 |          79,800 |         0.224 |            2.80 |                  0
     800 |         319,600 |         1.143 |            3.58 |                  0
    1537 |       1,180,416 |         4.248 |            3.60 |                  1

Projection with the measured cost per pair (short documents, so real texts will be slower):
  N =     1,000:            499,500 pairs -> about              2 s  (0.0 hours)
  N =    10,000:         49,995,000 pairs -> about            180 s  (0.0 hours)
  N =   100,000:      4,999,950,000 pairs -> about         17,993 s  (5.0 hours)
  N = 1,000,000:    499,999,500,000 pairs -> about      1,799

### Step 11: Evaluation helper for the LSH phase
This is the exact comparison the project statement asks for (precision, recall, F1, candidate reduction). It is tested here on brute force against itself so the LSH notebook can reuse it unchanged.

- **Candidate pairs** = what LSH returns before verification. Recall here measures *how many truly similar pairs LSH kept*.
- **Verified pairs** = candidates that also pass exact Jaccard $\ge$ threshold. Precision of verified pairs is 1.0 by construction; precision of the raw candidates shows how much wasted work LSH did.

In [12]:
def evaluate_pairs(found_pairs, truth_pairs, n_docs: int) -> Dict[str, float]:
    """found_pairs / truth_pairs: iterables of (doc_a, doc_b). Order inside a pair does not matter."""
    norm = lambda p: tuple(sorted((p[0], p[1])))
    F, T = {norm(p) for p in found_pairs}, {norm(p) for p in truth_pairs}
    tp, fp, fn = len(F & T), len(F - T), len(T - F)
    precision = tp / (tp + fp) if (tp + fp) else 1.0
    recall = tp / (tp + fn) if (tp + fn) else 1.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    total = n_docs * (n_docs - 1) // 2
    return {"tp": tp, "fp": fp, "fn": fn, "precision": precision, "recall": recall, "f1": f1,
            "candidate_pairs": len(F), "total_pairs": total, "candidate_reduction": 1 - len(F) / total if total else 0.0}


def speedup(brute_force_seconds: float, lsh_seconds: float) -> float:
    return brute_force_seconds / lsh_seconds if lsh_seconds > 0 else float("inf")


truth = [(p["doc_a"], p["doc_b"]) for p in res_sim["similar_pairs"]]
check = evaluate_pairs(truth, truth, sim_index.n_docs)
assert check["precision"] == 1.0 and check["recall"] == 1.0 and check["f1"] == 1.0
half = evaluate_pairs(truth[: len(truth) // 2], truth, sim_index.n_docs) if truth else None
print("Brute force vs itself :", {k: round(v, 3) if isinstance(v, float) else v for k, v in check.items()})
if half:
    print("Half of the pairs only :", {k: round(v, 3) if isinstance(v, float) else v for k, v in half.items()})

Brute force vs itself : {'tp': 4, 'fp': 0, 'fn': 0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'candidate_pairs': 4, 'total_pairs': 136, 'candidate_reduction': 0.971}
Half of the pairs only : {'tp': 2, 'fp': 0, 'fn': 2, 'precision': 1.0, 'recall': 0.5, 'f1': 0.667, 'candidate_pairs': 2, 'total_pairs': 136, 'candidate_reduction': 0.985}


### Summary and handoff to the LSH module

| Item | Value to use in the LSH notebook |
|---|---|
| Representation | k-word shingle **sets**, built by `preprocess_text` + `build_shingles` with the **same `ShingleConfig`** |
| Near-duplicate task | `NEAR_DUP_CFG` (k=3), threshold **0.80**, data: `eval_corpus.json`, truth: `ground_truth_near_duplicate.json` |
| Similarity task | `SIMILARITY_CFG` (k=1 + common/rare filter), threshold `SIMILARITY_THRESHOLD`, truth: `ground_truth_similarity.json` |
| Exact baseline | `BruteForceIndex` (`query`, `all_pairs`, `stats`) |
| Metrics | `evaluate_pairs` (precision, recall, F1, candidate reduction) and `speedup` |
| Experiments still to run with LSH | number of hash functions, bands $b$ x rows $r$, shingle size $k$, threshold, dataset size |

**Two points to state in the report**
1. Brute force is the exact baseline *under the chosen shingle representation and threshold*, not an absolute notion of truth.
2. Choosing LSH parameters means matching the S-curve $1-(1-s^r)^b$ to the threshold: the steep part of the curve should sit near 0.80 for near-duplicates, and near the similarity threshold for topical search.